# 01 · Generate model outputs into the shared cache

Needs a **GPU** runtime (Runtime → Change runtime type). A free T4 handles the small variants and 4-bit 7-8B models; use L4/A100 for full runs. Generation resumes from the cache after a disconnect.

Llama models are gated: add an `HF_TOKEN` secret (key icon in the sidebar) with access granted on Hugging Face.

Input: `items/items.jsonl` from notebook 03. Output: `generations/<model>/<checkpoint>/<lang>.jsonl`, one response per item per language, plus option probabilities for multiple-choice items.

In [ ]:
# Clone (or update) the repo and install it. Re-run after a runtime reset.
import os, subprocess
REPO_DIR = "/content/mpac"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hurshprasad/Multi-Agent-Diagnosis-of-Multilingual-Pluralistic-Alignment.git $REPO_DIR
else:
    !git -C $REPO_DIR pull --ff-only
%cd $REPO_DIR
!pip install -q -e ".[gpu]"
import sys
sys.path.insert(0, f"{REPO_DIR}/src")  # editable installs aren't always visible to a running kernel

In [ ]:
QUICK = True   # first run: 30 items, small model, minutes. Set False for the full ~500-item run.
MODEL = "qwen2.5-7b"   # full run only: "qwen2.5-7b" or "llama3.1-8b" (Llama needs HF_TOKEN). One model per session.
CHECKPOINT = "base"    # base | en_aligned | multi_aligned (adapters from notebooks/04_train_dpo.ipynb)
CONFIGS = ["configs/hi_pt.yaml"] + (["configs/hi_pt_quick.yaml"] if QUICK else [])
# Team run (en -> pt/es/it/de/fr): CONFIGS = ["configs/smoke.yaml"] or [] for the full default.

from google.colab import drive
drive.mount("/content/drive")
from mpac import load_config
from mpac.paths import resolve_root, subdir
cfg = load_config(*CONFIGS)
print("languages:", cfg["languages"], "| artifacts ->", resolve_root(cfg))

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("No HF_TOKEN secret set; gated models (Llama) will fail to download.")

In [ ]:
from mpac.config import all_languages, get_model
from mpac.data.loaders import load_items
from mpac.generation.cache import GenerationCache
from mpac.generation.generate import load_model, generate_for_language
from mpac.paths import subdir
from mpac.training.dpo import adapter_dir

items = load_items(cfg)   # built by notebooks/03_build_dataset.ipynb, verification applied
cache = GenerationCache(subdir(cfg, "generations"))
m = get_model(cfg, None if QUICK else MODEL)   # the quick config has only the 1.5B model
print("model:", m["id"], "->", m["hf_name"])
adapter = adapter_dir(cfg, m["id"], CHECKPOINT)
print("checkpoint:", CHECKPOINT, "| adapter:", adapter)
tok, model = load_model(m["hf_name"], load_in_4bit=cfg["generation"]["load_in_4bit"],
                        adapter_path=str(adapter) if adapter else None)
for lang in all_languages(cfg):
    n = generate_for_language(tok, model, items, model_id=m["id"], checkpoint=CHECKPOINT, lang=lang,
                              cache=cache, max_new_tokens=cfg["generation"]["max_new_tokens"],
                              batch_size=cfg["generation"]["batch_size"])
    print(lang, "wrote", n)

In [ ]:
from mpac.signals.acs import alignment_collapse_scores
gens = cache.load_all(m["id"], CHECKPOINT, all_languages(cfg))
acs = alignment_collapse_scores(gens, {it.item_id: it.objective for it in items})
acs   # only forced-choice items score without embeddings; add LaBSE embeddings for free text

## Free-text items: LaBSE embedding ACS

Helpfulness (UltraFeedback) and PLURAL prompts have no options, so they are scored by how far the target-language response moves from the English one in LaBSE space (1 − cosine). Reported **separately** from the forced-choice ACS above, because the two are on different scales.

- `prompt_floor`: the same divergence between the English prompt and its translation. That is what "same meaning, different language" scores for this language pair; read each row's ACS relative to it.
- Runs straight from the cache, so it doesn't need the model from cell 4 (you can run it after a runtime reset by running cells 1–2 first). Embeddings are cached under `embeddings/`.
- LaBSE reads the first 256 tokens of each response.

In [ ]:
!pip install -q "sentence-transformers>=3.0"
import pandas as pd
from mpac.config import all_languages, get_model
from mpac.data.loaders import load_items
from mpac.generation.cache import GenerationCache
from mpac.paths import subdir
from mpac.signals.acs import alignment_collapse_scores
from mpac.signals.embeddings import embed_generations, prompt_floor
from mpac.signals.lightweight import Embedder

items = load_items(cfg)
m = get_model(cfg, None if QUICK else MODEL)
gens = GenerationCache(subdir(cfg, "generations")).load_all(m["id"], CHECKPOINT, all_languages(cfg))
embedder = Embedder()   # sentence-transformers/LaBSE, on the GPU if there is one
emb = embed_generations(gens, embedder, subdir(cfg, "embeddings"))
free = alignment_collapse_scores(gens, {it.item_id: it.objective for it in items},
                                 embeddings=emb, signal="embedding")
table = pd.DataFrame(free).T
table["prompt_floor"] = pd.Series(prompt_floor(items, embedder, cfg["languages"]["targets"]))
table.round(3)